In [1]:
import requests, json, csv, sys
import pandas as pd
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from bs4 import BeautifulSoup
import warnings
warnings.filterwarnings("ignore")


In [2]:
n = 0

with open('result.txt', 'r') as f:
    for _ in range(n):
        next(f)
    data = f.read().splitlines()
print(n)

# 재시도 전략 설정
retries = Retry(total=10,  # 최대 재시도 횟수
                backoff_factor=1,  # 지연 시간에 대한 백오프 인자
                status_forcelist=[500, 502, 503, 504, 104, 10054, 2])  # 재시도할 상태 코드

# HTTPAdapter와 재시도 전략을 사용해 Session 객체 생성
session = requests.Session()
adapter = HTTPAdapter(max_retries=retries)
session.mount('http://', adapter)
session.mount('https://', adapter)

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
            'Accept-Language': 'ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7',}

fields = ['index', 'name', 'best_name', 'rating', 'rating_count', 'author', 'author_type', 
          'totalTime', 'recipeYield', 'recipeLev', 'image', 'description', 
          'datePublished', 'recipeIngredient', 'recipeInstructions']


def process(recipe_id):
    
    new_url = f'https://www.10000recipe.com/recipe/{recipe_id}'      
    new_response = session.get(new_url, headers=headers, verify=False)
        
    html = new_response.text
    soup = BeautifulSoup(html, 'html.parser')
    food_info = soup.find(attrs={'type':'application/ld+json'})
    food_lev = soup.find(attrs={'class':'view2_summary_info3'})
    reviews = soup.find_all(attrs={'class': 'media-heading'})
    food_best_tit = soup.find(attrs={'style':'color:#74b243;'})

    total_rating = 0
    review_count = len(reviews)

    for review in reviews:
        star_images = review.find_all('img')
        total_rating += len(star_images)

    # 평균 평점 계산
    if review_count > 0:
        average_rating = total_rating / review_count
    else:
        average_rating = 0
    
    try:
        result = json.loads(food_info.text, strict=False)
        try:    
            result['recipeIngredient'] = '|'.join(result['recipeIngredient'])  # '|'를 구분자로 사용
        except:
            food_ingr = soup.find('div', class_='cont_ingre').find('dd').text
            result['recipeIngredient']= '|'.join(food_ingr.split(' ,'))
        
        instructions = [step['text'] for step in result['recipeInstructions']]
        result['recipeInstructions'] = '|'.join(instructions)  # '|'를 구분자로 사용
        result['image'] = '|'.join(result['image'])
        result['recipeLev'] = food_lev.text
        result['index'] = recipe_id
        result['author_type'] = result['author']['@type'].lower()
        result['author'] = result['author']['name']
        result['rating'] = average_rating
        result['rating_count'] = review_count
        result['best_name'] = food_best_tit.text
        
        row = {field: result.get(field, '') for field in fields}
        return row
    except (AttributeError, KeyError, json.JSONDecodeError):
        return recipe_id
        
try:
    with ThreadPoolExecutor(max_workers=30) as executor, \
         open('recipe_6_14.csv', 'w', newline='', encoding='utf-8') as csvfile, \
         open('except.txt', 'w') as txtfile:
    
        writer = csv.DictWriter(csvfile, fieldnames=fields)
        if n == 0:
            writer.writeheader()
            n += 1    
        
        futures = [executor.submit(process, id) for id in data]
        for future in tqdm(as_completed(futures), total=len(data)):
            result = future.result()
            if isinstance(result, dict):
                writer.writerow(result)
            else:
                txtfile.write(str(result)+'\n')
except KeyboardInterrupt:
    sys.exit(0)

0


100%|██████████| 209142/209142 [3:40:06<00:00, 15.84it/s]   


In [3]:
df = pd.read_csv("recipe_6_14.csv")

In [4]:
df['cat4'] = None #종류별
df['cat3'] = None #상황별
df['cat2'] = None #재료별

In [5]:
cat4 = {63:'밑반찬', 56:'메인반찬', 54:'국/탕', 55:'찌개', 60:'디저트',
       53:'면/만두', 52:'밥/죽/떡', 61:'퓨전', 57:'김치/젓갈/장류', 58:'양념/소스/잼',
       65:'양식', 64:'샐러드', 68:'스프', 66:'빵', 69:'과자',
       59:'차/음료/술', 62:'기타'}

cat2 = {12:'일상', 18:'초스피드', 13:'손님접대', 19:'술안주', 21:'다이어트',
        15:'도시락', 17:'영양식', 17:'간식', 45:'야식', 20:'푸드스타일링',
        46:'해장', 44:'명절', 14:'이유식', 22:'기타'}

cat3 = {70:'소고기', 71:'돼지고기', 72:'닭고기', 23:'육류', 28:'채소류',
        24:'해물류', 50:'달걀/유제품', 33:'가공식품류', 47:'쌀', 32:'밀가루',
        25:'건어물류', 31:'버섯류', 48:'과일류', 27:'콩/견과류', 26:'곡류',
        34:'기타'}

In [6]:
def count_page(catego):
    new_url = 'https://www.10000recipe.com/recipe/list.html?'+catego
    new_url += '&order=reco&page=1'
    
    new_response = session.get(new_url, headers=headers, verify=False)
    html = new_response.text
    soup = BeautifulSoup(html, 'html.parser')
    count = int((soup.find(attrs={'class':"m_list_tit"}).find('b').text).replace(',',''))
    count = int(count/40)+1
    
    return list(i+1 for i in range(count))

In [7]:
def process4(catego, i, k):
    recipe_ids = []
    new_url = 'https://www.10000recipe.com/recipe/list.html?'+catego
    new_url += '&order=reco&page='+f'{k}'
    new_response = session.get(new_url, headers=headers, verify=False)
    html = new_response.text
    soup = BeautifulSoup(html, 'html.parser')
    for link in soup.find_all('a', class_='common_sp_link'):
        href = link['href']
        recipe_id = href.split('/')[2]
        recipe_ids.append(recipe_id)
            
    for recipe_id in recipe_ids:
        if int(recipe_id) in df['index'].values:
            df.loc[df['index'] == int(recipe_id), 'cat4'] = cat4[i]
            

In [8]:
def process2(catego, i, k):
    recipe_ids = []
    new_url = 'https://www.10000recipe.com/recipe/list.html?'+catego
    new_url += '&order=reco&page='+f'{k}'
    new_response = session.get(new_url, headers=headers, verify=False)
    html = new_response.text
    soup = BeautifulSoup(html, 'html.parser')
    for link in soup.find_all('a', class_='common_sp_link'):
        href = link['href']
        recipe_id = href.split('/')[2]
        recipe_ids.append(recipe_id)
            
    for recipe_id in recipe_ids:
        if int(recipe_id) in df['index'].values:
            df.loc[df['index'] == int(recipe_id), 'cat2'] = cat2[i]
            

In [9]:
def process3(catego, i, k):
    recipe_ids = []
    new_url = 'https://www.10000recipe.com/recipe/list.html?'+catego
    new_url += '&order=reco&page='+f'{k}'
    new_response = session.get(new_url, headers=headers, verify=False)
    html = new_response.text
    soup = BeautifulSoup(html, 'html.parser')
    for link in soup.find_all('a', class_='common_sp_link'):
        href = link['href']
        recipe_id = href.split('/')[2]
        recipe_ids.append(recipe_id)
            
    for recipe_id in recipe_ids:
        if int(recipe_id) in df['index'].values:
            df.loc[df['index'] == int(recipe_id), 'cat3'] = cat3[i]
            

In [10]:
with ThreadPoolExecutor(max_workers=30) as executor:
    for i in cat4:
        catego = f'cat4={i}'
        data = count_page(catego)
        futures = [executor.submit(process4, catego, i, k) for k in data]
        for future in tqdm(as_completed(futures), total=len(data)):
            future.result()
    
    for i in cat2:
        catego = f'cat2={i}'
        data = count_page(catego)
        futures = [executor.submit(process2, catego, i, k) for k in data]
        for future in tqdm(as_completed(futures), total=len(data)):
            future.result()
            
    for i in cat3:
        catego = f'cat3={i}'
        data = count_page(catego)
        futures = [executor.submit(process3, catego, i, k) for k in data]
        for future in tqdm(as_completed(futures), total=len(data)):
            future.result()

100%|██████████| 296/296 [00:16<00:00, 17.72it/s]


In [11]:
df.to_csv("to_be_final_crawling.csv", index=False)

In [1]:
df['cat3'].isna().sum()

NameError: name 'df' is not defined